# The Cumulative Distribution Function (CDF)

This notebook introduces the **cumulative distribution function** — the probability accumulated up to each point, $F_X(x) = P(X \leq x)$. It turns area calculations into simple height lookups, and it works for both discrete and continuous distributions. It is based on the eighth video of Lesson 2 (Probability Distributions).


## 1. Introduction

To find the probability that a call lasts between 2 and 3 minutes with a PDF, you must compute an **area** — not convenient. The CDF stores, at each point $x$, the probability that the call takes **at most** $x$ minutes. Probabilities become **heights** you can read off directly.

After completing this notebook you will be able to:

- Build the cumulative probabilities of a discrete distribution and plot its staircase CDF.
- Build the continuous CDF as accumulated area under the PDF.
- State the four properties every CDF satisfies, and read probabilities from CDF heights.


## 2. Theory

### Cumulative probability

The cumulative probability at a reference point is the probability that the event happened **before (or at)** that point:

$$
F_X(x) = P(X \leq x)
$$

- Capital $F$ — lowercase $f$ is reserved for the density. Subscript for the variable: $F_X$, or $F_Y$ for a variable $Y$.
- Defined for **every** real number from $-\infty$ to $+\infty$.

### The discrete case: a staircase

For the 1-minute-window call center: $F(1) = P(0\text{-}1)$; $F(2)$ adds the second bar; ... ; $F(5) = 1$. The CDF **jumps** at every possible outcome — the height of the jump is exactly $p(x)$, the PMF at that point — and is **flat** in between, where no outcome accumulates probability.

### The continuous case: a smooth climb

Sweep along the PDF from the left, plotting accumulated **area**: the CDF starts at 0 (no area yet), ends at 1 (all of it), and is smooth — no point carries mass, so there are no jumps. If the PDF extends infinitely, the CDF *limits* to 0 on the left and 1 on the right.

The key relationship: the height of the CDF at $x$ **equals** the area under the PDF up to $x$. If $P(X \leq 2) = \frac{1}{2}$, the CDF passes through height $\frac{1}{2}$ at $x = 2$.

### The four properties

1. $0 \leq F_X(x) \leq 1$ — it is a probability.
2. Left endpoint (or limit): **0**.
3. Right endpoint (or limit): **1**.
4. **Never decreasing** — it accumulates non-negative probabilities.


## 3. Worked Example

Using the discrete call-center table (windows of 1 minute):

| Window | $p$ | Cumulative $F$ |
|---|---|---|
| 0-1 | 0.15 | 0.15 |
| 1-2 | 0.35 | 0.50 |
| 2-3 | 0.30 | 0.80 |
| 3-4 | 0.15 | 0.95 |
| 4-5 | 0.05 | 1.00 |

$F$ climbs from 0 to 1, and useful probabilities drop out by subtraction: $P(2 < X \leq 4) = F(4) - F(2) = 0.95 - 0.50 = 0.45$ — no re-adding of bars needed.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

# numpy compatibility: trapezoid was renamed from trapz in numpy 2.0
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz


### Discrete: cumulative sums


In [ ]:
p = np.array([0.15, 0.35, 0.30, 0.15, 0.05])   # PMF per 1-min window
F = np.cumsum(p)

print("window  p(x)   F(x)")
for i, (pi, Fi) in enumerate(zip(p, F)):
    print(f"{i}-{i+1}     {pi:.2f}   {Fi:.2f}")
print(f"\nP(2 < X <= 4) = F(4) - F(2) = {F[3]:.2f} - {F[1]:.2f} = {F[3]-F[1]:.2f}")


### Continuous: the CDF as accumulated area

The same gamma-shaped call PDF as before; its CDF is the running integral, computed with a cumulative trapezoid sum.


In [ ]:
xs = np.linspace(0, 5, 1_000)
shape, scale = 4.0, 0.55
pdf = xs ** (shape - 1) * np.exp(-xs / scale)
pdf /= np.trapezoid(pdf, xs)

dx = xs[1] - xs[0]
cdf = np.concatenate([[0], np.cumsum((pdf[1:] + pdf[:-1]) / 2 * dx)])

for t in [1, 2, 3, 5]:
    print(f"F({t}) = P(X <= {t}) = {np.interp(t, xs, cdf):.3f}")
print(f"P(2 <= X <= 3) = F(3) - F(2) = "
      f"{np.interp(3, xs, cdf) - np.interp(2, xs, cdf):.3f}  (area, via heights!)")


### The four properties, checked


In [ ]:
print(f"1. all values in [0,1]:  {bool(((cdf >= 0) & (cdf <= 1.0001)).all())}")
print(f"2. left endpoint:        F(0) = {cdf[0]:.3f}")
print(f"3. right endpoint:       F(5) = {cdf[-1]:.3f}")
print(f"4. never decreasing:     {bool((np.diff(cdf) >= -1e-12).all())}")


## 5. Visualizations

### Discrete: PMF bars and the staircase CDF

Jump heights in the right panel equal the bar heights on the left; the flat stretches are values that accumulate nothing.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(np.arange(5) + 0.5, p, width=1.0, color="#4e79a7", edgecolor="white")
axes[0].set_title("PMF: probability per window")
axes[0].set_xlabel("Call duration (min)")
axes[0].set_ylabel("p(x)")

# staircase: F jumps at the end of each window
xs_step = np.arange(0, 6)
ys_step = np.concatenate([[0], F])
axes[1].step(xs_step, ys_step, where="post", color="#e15759", linewidth=2)
for i, Fi in enumerate(F):
    axes[1].plot(i + 1, Fi, "o", color="#e15759")
axes[1].axhline(1, color="gray", linestyle=":")
axes[1].set_title("CDF: jumps = PMF values, flats between")
axes[1].set_xlabel("Call duration (min)")
axes[1].set_ylabel("F(x)")

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Continuous: PDF area = CDF height

The shaded area up to $x = 2$ on the left *is* the marked height on the right — two views of the same information: areas on the left, heights on the right.


In [ ]:
t = 2.0
Ft = np.interp(t, xs, cdf)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(xs, pdf, color="#2c5f8a", linewidth=2)
mask = xs <= t
axes[0].fill_between(xs[mask], pdf[mask], color="#4e79a7", alpha=0.5)
axes[0].set_title(f"PDF: area up to {t:.0f} min = {Ft:.2f}")
axes[0].set_xlabel("Call duration (min)")
axes[0].set_ylabel("f(x)")

axes[1].plot(xs, cdf, color="#e15759", linewidth=2)
axes[1].plot([t, t], [0, Ft], "k--", linewidth=1)
axes[1].plot([0, t], [Ft, Ft], "k--", linewidth=1)
axes[1].plot(t, Ft, "ko")
axes[1].annotate(f"F({t:.0f}) = {Ft:.2f}", (t, Ft),
                 textcoords="offset points", xytext=(10, -14), fontsize=11)
axes[1].set_title("CDF: the same number, read as a height")
axes[1].set_xlabel("Call duration (min)")
axes[1].set_ylabel("F(x)")

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Exercises

### Basic

1. From the worked table, read off $P(X \leq 3)$ and $P(X > 3)$.
2. Why can a CDF never decrease?

### Intermediate

3. For a fair die, write the CDF values $F(1), F(2), \dots, F(6)$, and sketch (mentally) where the jumps are and how big each is.
4. For the uniform distribution on $[0, 5]$, write $F(x)$ as an explicit formula and check $F(4.5) = 0.9$ (exercise 4 of the previous notebook, now in CDF form).

### Advanced

5. Compute the **empirical CDF** of 100,000 simulated calls (sort the samples; plot rank/n against value) and overlay it on the theoretical CDF — they should be indistinguishable.


## 7. Solutions


In [ ]:
# Exercise 1
print(f"1.  P(X<=3) = F(3) = {F[2]:.2f};  P(X>3) = 1 - F(3) = {1-F[2]:.2f}")

# Exercise 2
print("2.  It accumulates probabilities, which are >= 0 — each step adds, never removes.")

# Exercise 3
die_F = np.cumsum([1/6] * 6)
print("3.  F(k) for k=1..6:", [f"{v:.3f}" for v in die_F],
      "— jumps of 1/6 at each face, flat in between.")

# Exercise 4
print("4.  F(x) = 0 for x<0;  x/5 for 0<=x<=5;  1 for x>5.  F(4.5) =", 4.5 / 5)

# Exercise 5: empirical vs theoretical CDF
samples = rng.gamma(shape=4.0, scale=0.55, size=100_000)
samples = np.sort(samples[samples <= 5])
ecdf_y = np.arange(1, len(samples) + 1) / len(samples)

plt.figure(figsize=(8, 4.5))
plt.plot(samples, ecdf_y, color="#4e79a7", linewidth=3, alpha=0.6,
         label="empirical CDF (100k samples)")
plt.plot(xs, cdf, color="#e15759", linewidth=1.5, linestyle="--",
         label="theoretical CDF")
plt.title("Empirical and theoretical CDFs coincide")
plt.xlabel("Call duration (min)")
plt.ylabel("F(x)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
print("5.  The curves overlap — the ECDF converges to the true CDF.")


## 8. Conclusion

- $F_X(x) = P(X \leq x)$: capital $F$, defined on all of $\mathbb{R}$, turning **areas into heights**.
- Discrete CDFs are **staircases** — jump sizes are the PMF; continuous CDFs are **smooth climbs** — slopes come from the PDF.
- Four properties: values in $[0,1]$, limits 0 (left) and 1 (right), never decreasing.
- Interval probabilities by subtraction: $P(a < X \leq b) = F(b) - F(a)$ — often the most convenient route, which is why knowing both PDF and CDF pays off.
